# Lab 12 — Multi-Agent Systems & Self-Reflection

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- state the **honest default** (one agent, one context) and the **three pressures** that justify a team — context separation, role specialisation, parallelism,
- name the coordination patterns (**supervisor-workers, peer handoffs, debate/critique**) and the anti-patterns (**agent sprawl, telephone game, ~15x cost, shared-state races**),
- build the **minimal viable team**: a writer plus a **source-grounded critic** in a fresh context, with **checkable verdicts** and a **bounded** revise loop (Slide 17),
- **measure** the critic against a single-agent baseline — grounding (unsupported-claim rate), **tokens**, and **latency** — reusing S11-style cost accounting,
- explain why **external ground truth** (the retrieved sources) is what makes reflection work, and why *intrinsic* self-correction often does not,
- observe the **telephone game** first-hand in a two-worker parallel merge.

> ⏱️ Estimated time: 90–120 minutes. The corpus is tiny and offline; every fact in it is
> invented, so a correct sourced claim can only have come from a page the agent actually read.

## Theory recap — teams that earn their place, and critics that need a signal

### The honest default: one agent

The default architecture is **one loop, one context, all tools** — it covers most production use
cases. Its strength is a property we have leaned on all semester: the *full run history* sits in
one context window, so every decision is conditioned on complete information and **nothing is lost
in transfer**. Every multi-agent design sacrifices some of this by construction. Industry guidance
is blunt: start with the simplest design and add agents only when one *demonstrably* plateaus.
**Multi-agent is the option that must justify itself** — never the starting point.

### Three pressures that justify a team

Only three mechanical pressures survive scrutiny, each mapped to a hard limit of the single loop:

- **Context separation** — a worker gets a clean window; a 40k-token document is distilled to a
  half-page of findings that never touches the lead's context. Sub-agents are *rented context*.
- **Role specialisation** — a focused system prompt is followed more reliably than one prompt
  juggling planner, searcher and critic at once (S04). A *role* is a prompt + a tool set + a
  defined input/output — **not a different model**; usually the same model in a different context.
- **Parallelism** — independent subtasks run concurrently, collapsing wall-clock latency. The
  qualifier is load-bearing: parallelism only pays when the subtasks are genuinely **independent**.

### Coordination patterns

**Supervisor-workers** (orchestrator-workers): one lead decomposes the goal, spawns workers with
scoped *briefs*, collects compressed findings, and synthesises — centralised control, one place to
trace and budget, one single point of failure. The dominant production pattern. **Peer handoffs**:
no boss; each agent finishes its part and passes control plus state to the next (triage →
specialist) — control travels with the conversation; suits routing. **Debate / critique**: agents
argue positions or review each other's drafts; measurable factuality gains on some tasks
(Du et al., 2023), at multiplied cost. Frameworks (CrewAI, AutoGen, LangGraph) express these same
concepts; the concepts transfer, the APIs do not (S07).

### Anti-patterns and the price of a team

Four failure modes each earn their name: **agent sprawl** (a component that never changes control
flow is a *function*, not an agent), the **telephone game** (every handoff is a lossy natural-
language compression — a researcher's *"preliminary, small sample, possible effect"* becomes the
report's flat *"effect"*), **cost explosion** (Anthropic 2025 measured a single agent at ~4x chat
tokens and a **multi-agent system at ~15x**), and **shared-state races** (parallel writers on one
document interleave nondeterministically). The MAST taxonomy (Cemri et al., 2025) traces most
failures to **system design** — vague specifications, inter-agent misalignment, missing
verification — **not raw model capability**. Evidence sorts along **task coupling**: teams win on
loosely coupled, breadth-first work; they lose where subtasks are tightly coupled.

### The minimal viable team: writer + critic

If most multi-agent *value* is independent verification and most *cost* is coordination, the
smallest team worth building is **one writer plus one critic** — almost all of the second pair of
eyes, none of the machinery. Three load-bearing details: the critic runs in a **fresh context**
(it sees the task and draft, never the writer's reasoning — independence is the whole mechanism),
its verdict must be **checkable** (list every claim lacking a source; not *"is this good?"*), and
the loop is **bounded** by `max_rounds` (the S02 stop condition; two is usually right).

### Self-reflection and its limits

**Reflexion** (Shinn et al., 2023) is reinforcement learning where the policy update is *a
paragraph of English*: an actor attempts, an evaluator scores, a reflection step writes a verbal
lesson, and an episodic memory prepends it to the next attempt — no weights change. It reported
strong gains (91% pass@1 on HumanEval vs. 80% baseline) — **but only because the evaluator was
reliable** (unit tests, environment signals). The cold shower: **intrinsic self-correction**,
where a model critiques its own reasoning with no external signal, *often reduces accuracy*
(Huang et al., 2024), because the second pass samples the same distribution that produced the
error — **shared blind spots**. The resolution: reflection is an **amplifier of its feedback
signal**. Wire it to external ground truth — **executable checks**, **retrieval/grounding**
(CRITIC, Gou et al., 2024), **environment outcomes**, humans — and it amplifies correction; feed
it self-opinion and it amplifies the blind spots. Self-critique catches what the model *knows but
did not apply*, never what it *does not know*.

### This lab

You will take the research agent, run a **baseline** report on three topics (keeping S11-style
token/latency traces), add a **source-grounded critic** in a fresh context with checkable verdicts
and a two-round cap, **measure** grounding/tokens/latency before vs. after, and finally split the
work across **two parallel workers** to hunt the telephone game in the merge — exactly the lab the
lecture announced (Slide 23).

## Part A — Setup, connectivity & cost accounting

Dependencies stay minimal: `ollama`, `pandas`, `matplotlib`. One local chat model does every
role — writer and critic differ only in **context and prompt**, not in weights, exactly as the
lecture insists.

We also build a tiny **token-accounting** helper. Session 11 made a single agent observable; this
week multiplies model calls, and *you can only manage what you can measure*. Every model call in
this lab records its token count into a shared `TRACE` list, so Part E can price the critic.

In [ ]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")

OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
    print(f"Ollama reachable — this lab will use {MODEL!r} for writer and critic.")
except Exception as exc:
    print("Ollama not reachable:", exc)
    print("Fix: start Ollama with `ollama serve`, then `ollama pull qwen2.5:7b`.")
    print("(You can still read the notebook; the LLM cells will simply be skipped.)")

# --- load the offline corpus (nine short invented pages across three topics) ---
DATA = Path("data") if Path("data").exists() else Path("../Lab12_MultiAgent/data")
with open(DATA / "corpus.json", "r", encoding="utf-8") as f:
    CORPUS = json.load(f)
print(f"\nLoaded {len(CORPUS)} offline pages across topics:",
      ", ".join(sorted({p["topic"] for p in CORPUS})))

# --- a global trace: every model call appends its token count here (S11) ---
TRACE = []

Ollama reports `prompt_eval_count` (input tokens it evaluated) and `eval_count`
(generated, billed tokens — *thinking included*). Our helper sums them so we can attribute cost
per span, the way Session 11 attributed cost per call.

In [ ]:
def count_tokens(resp):
    """Total tokens for one ollama.chat response = prompt + generated (S11 cost accounting)."""
    prompt = resp.get("prompt_eval_count", 0) or 0
    output = resp.get(___, 0) or 0   # generated (billed) tokens
    return prompt + output


def chat(system, user):
    """One model call with a system + user turn. Returns (text, token_count)."""
    resp = ollama.chat(model=MODEL, messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ])
    return resp["message"]["content"].strip(), count_tokens(resp)

<details>
<summary><b>Click here for the solution</b></summary>

```python
def count_tokens(resp):
    prompt = resp.get("prompt_eval_count", 0) or 0
    output = resp.get("eval_count", 0) or 0   # generated (billed) tokens
    return prompt + output
```

</details>

> **Q:** Why is a single agent with one context window the correct default architecture, and what property makes it so strong?
<details><summary>Click for answer</summary>

A single agent keeps the entire run history — every tool result, error, and intermediate
decision — in one context window, so every decision is conditioned on full information. There is
**no inter-component information transfer and therefore no transfer loss**. Multi-agent designs
sacrifice some of this property by construction, so they must compensate with a benefit (context
separation, specialisation, or parallelism) that the specific task actually needs.
</details>

> **Q:** Name the three pressures that genuinely justify a team and the hard limit of the single loop each one addresses.
<details><summary>Click for answer</summary>

**Context separation** addresses the finite context window: bulky raw material lives in worker
contexts and only compressed findings return. **Role specialisation** addresses degrading
instruction-following in long, multi-purpose prompts: narrow prompts with narrow tool sets are
followed more reliably. **Parallelism** addresses sequential latency: independent subtasks run
concurrently, collapsing wall-clock time even though token cost stays the same or rises.
</details>

> **📝 Report task R1 — one agent or many? (deliverable):** The lecture insists the single agent is the default and that multi-agent must *earn its place* via one of three pressures — **context separation, role specialisation, parallelism**. For **your** research agent, decide whether a *team* (parallel sub-researchers) is justified, and whether an *agent-plus-critic* is justified. Name which of the three pressures (if any) each design targets, and state the burden-of-proof argument in your own words.
>
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part B — The research writer (single-agent baseline)

The running example: the research agent drafts a Markdown report from retrieved pages. For this
lab, *retrieval is already done* — for each topic we simply hand the writer the relevant pages
(an offline stand-in for search + fetch). The writer's job is **synthesis**: read the sources,
write a `## Findings` section that cites source numbers, and a `## Sources` list.

This is the single-agent baseline the lecture asks you to keep as a **control group** (Slide 23,
step 1). Every writer call records its tokens into `TRACE`.

In [ ]:
def pages_for(topic):
    """The retrieved pages for one topic (offline stand-in for search+fetch)."""
    return [p for p in CORPUS if p["topic"] == topic]


WRITER_SYSTEM = (
    "You are a research writer. Write a concise Markdown report answering the question, "
    "using ONLY the numbered sources provided. Cite the source number in square brackets "
    "after each claim, e.g. [2]. The report MUST contain a '## Findings' section and a "
    "'## Sources' section. Do not invent facts that are not in the sources."
)


def write_report(topic, question):
    """Single-agent baseline: one writer call, no critic."""
    pages = pages_for(topic)
    # number the sources so the writer (and later the critic) can cite them
    src_block = "\n".join(f"[{i}] {p['title']}: {p['body']}" for i, p in ___)
    user = f"Question: {question}\n\nSources:\n{src_block}\n\nWrite the report."
    resp = ollama.chat(model=MODEL, messages=[
        {"role": "system", "content": WRITER_SYSTEM},
        {"role": "user", "content": user},
    ])
    TRACE.append({"span": "write", "topic": topic, "tokens": ___})   # cost accounting (S11)
    return resp["message"]["content"].strip()


if OLLAMA_OK:
    topic = "zephyr7"
    baseline = write_report(topic, TOPICS := {
        "zephyr7": "What does the evidence say about the ZEPHYR-7 planning benchmark, "
                   "including multi-agent vs single-agent scores and their cost?",
        "halcyon": "How does the HALCYON memory store prevent shared-state races, "
                   "and what are its limitations?",
        "corvus":  "What is the CORVUS critic protocol and what grounding results does it report?",
    }[topic])
    print(baseline[:600], "...")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    src_block = "\n".join(f"[{i}] {p['title']}: {p['body']}" for i, p in enumerate(pages, 1))
    ...
    TRACE.append({"span": "write", "topic": topic, "tokens": count_tokens(resp)})
```

`enumerate(pages, 1)` numbers the sources from 1 so the citations `[1]`, `[2]`, … line up with the
list the critic will later receive; `count_tokens(resp)` records the writer's cost into the shared
trace, exactly as Session 11 attributed cost per model call.

</details>

> **Q:** In the scaled-out research agent, what does a *brief* contain, and why does the architecture stand or fall on worker-side compression?
<details><summary>Click for answer</summary>

A brief is the worker's contract: the subtopic, the permitted tools, the intended effort/budget,
and the required output format. The lead never sees raw pages — it reads only the workers'
returned **findings**. If those findings are not faithful, dense compressions of what was read,
the lead synthesises from distorted inputs and the pipeline launders away nuance; **compression
quality is therefore the load-bearing element**.
</details>

## Part C — Add a critic: the minimal viable team

Now the code slide (Slide 17). The critic is the **same model in a fresh context**: it sees only
the draft and the numbered sources — never the writer's reasoning — and returns a **checkable
verdict** (`pass`/`revise`) plus concrete **issues**. The loop is **bounded** to `MAX_ROUNDS`:
on `pass` we accept; when rounds run out we do **not** loop forever and do **not** silently ship —
we return the last draft flagged as unresolved.

First the critic's system prompt. This is **report task R4** — no fold-out solution.

In [ ]:
CRITIC_SYSTEM = (
    "You are a strict fact-checking critic. You see ONLY a draft report and a numbered "
    "list of source snippets. You did NOT see how the draft was written.\n"
    ___
)

> **📝 Report task R4 — complete the checkable-verdict critic prompt:** The cell above builds the critic's system prompt. Complete the `___` so the critic (i) sees only the draft and the numbered sources, (ii) returns **strict JSON** with a `verdict` of `"pass"`/`"revise"` and a list of concrete `issues`, and (iii) is told to flag **every claim no source supports** and **every missing required section** — checkable issues, not a vibe rating. Justify each design choice against the lecture (fresh context, external signal, checkable verdicts).
>
> *No solution is provided — complete the cell above and include your prompt plus the justification in your lab report.*

Now the critic call and the bounded reflection loop. The critic must return
**strict JSON**; we parse it defensively (a malformed review forces another round rather than a
false `pass`). The writer then revises **with the critic's issues in context**.

In [ ]:
MAX_ROUNDS = 2   # Slide 17: two rounds max; gains beyond that are noise


def critique(draft, pages):
    """Fresh-context critic: sees only the draft + numbered sources (the external signal)."""
    src_block = "\n".join(f"[{i}] {p['title']}: {p['body']}" for i, p in enumerate(pages, 1))
    user = f"DRAFT:\n{draft}\n\nSOURCES:\n{src_block}"
    resp = ollama.chat(model=MODEL, messages=[
        {"role": "system", "content": CRITIC_SYSTEM},
        {"role": "user", "content": user},
    ], format="json")
    TRACE.append({"span": "critique", "tokens": count_tokens(resp)})
    try:
        review = json.loads(resp["message"]["content"])
    except Exception:
        review = {"verdict": "revise", "issues": ["critic returned unparseable JSON"]}
    return review


def revise(topic, question, draft, issues):
    """The writer revises its draft given the critic's concrete issues."""
    pages = pages_for(topic)
    src_block = "\n".join(f"[{i}] {p['title']}: {p['body']}" for i, p in enumerate(pages, 1))
    user = (f"Question: {question}\n\nSources:\n{src_block}\n\n"
            f"Your previous draft:\n{draft}\n\n"
            f"A critic raised these issues:\n- " + "\n- ".join(issues) +
            "\n\nRevise the report to fix EVERY issue. Keep only claims a source supports.")
    resp = ollama.chat(model=MODEL, messages=[
        {"role": "system", "content": WRITER_SYSTEM},
        {"role": "user", "content": user},
    ])
    TRACE.append({"span": "revise", "topic": topic, "tokens": count_tokens(resp)})
    return resp["message"]["content"].strip()


def write_with_critic(topic, question):
    """The minimal viable team: draft -> (critique -> revise) x MAX_ROUNDS."""
    pages = pages_for(topic)
    draft = write_report(topic, question)
    for round_i in range(___):                 # bounded: two rounds max (Slide 17)
        review = critique(draft, pages)
        verdict = review.get("verdict", ___)      # unparseable review -> force another look
        if verdict == ___:                          # critic is satisfied -> accept
            return draft, review, round_i
        draft = revise(topic, question, draft, review.get("issues", []))
    # rounds exhausted: return flagged, never silently shipped
    return draft, {"verdict": "unresolved", "issues": review.get("issues", [])}, MAX_ROUNDS


if OLLAMA_OK:
    q = {"zephyr7": "What does the evidence say about the ZEPHYR-7 planning benchmark, "
                    "including multi-agent vs single-agent scores and their cost?"}["zephyr7"]
    critiqued, review, rounds = write_with_critic("zephyr7", q)
    print(f"Stopped after {rounds} round(s); final verdict: {review['verdict']}")
    print(critiqued[:600], "...")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    for round_i in range(MAX_ROUNDS):                 # bounded: two rounds max (Slide 17)
        review = critique(draft, pages)
        verdict = review.get("verdict", "revise")      # unparseable review -> force another look
        if verdict == "pass":                          # critic is satisfied -> accept
            return draft, review, round_i
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`range(MAX_ROUNDS)` is the **stop condition** from Session 02 applied to a reflection loop: without
it, critique↔revise can oscillate or plateau while burning tokens. Defaulting `verdict` to
`"revise"` makes the loop **fail safe** — a critic response we could not parse triggers another
look rather than a false accept. Accepting only on `"pass"` means the writer must actually satisfy
the critic; if it never does within `MAX_ROUNDS`, we return the draft tagged `"unresolved"` instead
of silently shipping it (the lecture's `flag_unresolved`).

</details>

> **Q:** Why must the critic run in a *fresh context* that never sees the writer's reasoning?
<details><summary>Click for answer</summary>

A critic conditioned on the same context that produced the draft samples from a **correlated
distribution**: it inherits the drafter's framing and blind spots, so its review is a re-roll
rather than an independent check. A fresh context seeing only the task, the draft, and the sources
approximates independent review. **The critic's value comes precisely from what it does not see.**
</details>

> **Q:** Why must the critic's output be *checkable issues* rather than a quality judgement, and why is `MAX_ROUNDS` essential?
<details><summary>Click for answer</summary>

A vague prompt (*"is this good?"*) elicits vague approval, which drives no revision; concrete
demands — list every claim lacking a source, name every missing section — produce issues the
writer can act on and the system can re-verify. `MAX_ROUNDS` is the S02 **stop condition** reborn:
critique-revise can oscillate or plateau, so an uncapped loop burns budget without converging; an
expired loop should return the draft **flagged as unresolved**, not silently shipped.
</details>

## Part D — The external signal: a grounding metric

The critic only works because we give it **external ground truth** — the sources. Let us measure
the same thing mechanically, so Part E can compare baseline vs. critic **numerically**.

Our proxy for *grounding* is the **unsupported-claim rate**: split the report's `## Findings` into
claim-like sentences, and count how many share *no* meaningful keyword with the source text. It is
crude (keyword overlap, not entailment) — a lab stand-in for the retrieval-grounded check the
lecture describes (CRITIC, Gou et al., 2024) — but it moves in the right direction: hedged,
source-faithful reports score low; confident, invented ones score high.

In [ ]:
STOP = set("the a an of to and or in on for is are was were with that this it as by from "
               "at be can will may about into their its our we you they not no more most "
               "than then so such which who what how when where report sources findings".split())


def keywords(sentence):
    """Content words of a sentence, lowercased, minus stopwords and short tokens."""
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9\-]+", sentence.lower())
    return [w for w in words if w not in STOP and len(w) > 3]


def claim_sentences(report):
    """Sentences from the Findings section that look like factual claims."""
    body = report.split("## Findings", 1)[-1].split("## Sources", 1)[0]
    sents = re.split(r"(?<=[.!?])\s+", body)
    return [s.strip() for s in sents if len(keywords(s)) >= 2]


def unsupported_rate(report, pages):
    """Fraction of claim sentences whose content words never appear in any source."""
    sources_text = " ".join(p["title"] + " " + p["body"] for p in pages).lower()
    claims = claim_sentences(report)
    if not claims:
        return 0.0, 0
    unsupported = sum(1 for c in claims if not any(kw in sources_text for kw in ___))
    return unsupported / len(claims), len(claims)


if OLLAMA_OK:
    pages = pages_for("zephyr7")
    rate, n = unsupported_rate(critiqued, pages)
    print(f"Critiqued report: {n} claims, unsupported rate = {rate:.0%}")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    unsupported = sum(1 for c in claims if not any(kw in sources_text for kw in keywords(c)))
```

For each claim sentence, `keywords(c)` extracts its content words; the claim counts as *supported*
if **any** of them appears in the concatenated source text, and *unsupported* otherwise. This is a
deliberately simple keyword-overlap proxy — the point is a reproducible number that drops when the
critic forces the writer to remove or hedge claims no page supports.

</details>

## Part E — Measure: baseline vs. critic across three topics

Now the experiment the lecture asks for (Slide 23, step 4): run **all three topics** through both
the single-agent **baseline** and the **critic** pipeline, and compare on three axes —
**grounding** (unsupported-claim rate), **tokens**, and **latency**. `TRACE` gives us tokens per
span; we time each pipeline with `time.perf_counter`. No gaps here beyond what you have already
built — just run it and read the table.

In [ ]:
def run_pipeline(fn, topic, question):
    """Time one pipeline (fn returns a report string) and sum its trace tokens."""
    TRACE.clear()
    t0 = time.perf_counter()
    result = fn(topic, question)
    report = result[0] if isinstance(result, tuple) else result
    dt = time.perf_counter() - t0
    tokens = sum(e["tokens"] for e in TRACE)
    rate, n = unsupported_rate(report, pages_for(topic))
    return {"tokens": tokens, "latency_s": round(dt, 2),
            "unsupported": round(rate, 3), "claims": n, "report": report}


if OLLAMA_OK:
    rows = []
    for topic, question in TOPICS.items():
        base = run_pipeline(write_report, topic, question)
        crit = run_pipeline(write_with_critic, topic, question)
        rows.append({"topic": topic, "variant": "baseline", **{k: base[k] for k in
                     ("unsupported", "tokens", "latency_s", "claims")}})
        rows.append({"topic": topic, "variant": "critic", **{k: crit[k] for k in
                     ("unsupported", "tokens", "latency_s", "claims")}})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
else:
    df = pd.DataFrame()
    print("Ollama offline — run this cell with Ollama up to produce the comparison table.")

A quick visual: grounding and token cost, baseline vs. critic, averaged across
the three topics. This is the picture your R2 memo argues about.

In [ ]:
if OLLAMA_OK and not df.empty:
    agg = df.groupby("variant")[["unsupported", "tokens", "latency_s"]].mean()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
    agg["unsupported"].plot.bar(ax=ax1, color=["#003d86", "#9b0a7d"], rot=0)
    ax1.set_title("Unsupported-claim rate (lower = better)"); ax1.set_ylabel("fraction")
    agg["tokens"].plot.bar(ax=ax2, color=["#003d86", "#9b0a7d"], rot=0)
    ax2.set_title("Tokens (the price of the critic)"); ax2.set_ylabel("tokens")
    plt.tight_layout(); plt.show()

    prem = agg.loc["critic", "tokens"] / max(agg.loc["baseline", "tokens"], 1)
    print(f"Critic token premium: {prem:.2f}x the baseline "
          f"(the lecture's null hypothesis is that a team costs ~15x — a single critic is far cheaper).")
else:
    print("Nothing to plot yet — run Part E with Ollama up first.")

> **📝 Report task R2 — did the critic pay for itself? (the half-page memo):** This is the lecture's deliverable (Slide 23). Using the measured table from Part E — grounding (unsupported-claim rate), tokens, and latency, **baseline vs. critic** across the three topics — write a half-page memo answering one question: **did the critic pay for itself?** Argue with numbers, not impressions. State the token/latency premium you observed and the grounding change, and give the decision rule you would apply in production.
>
> *No solution is provided — include the half-page memo in your lab report.*

> **Q:** What token-cost multipliers did Anthropic (2025) report for agents and multi-agent systems relative to chat, and what mechanisms produce the multi-agent premium?
<details><summary>Click for answer</summary>

Roughly **4x** chat usage for a single agent and roughly **15x** for a multi-agent system,
measured on their own traffic. Mechanisms: each worker maintains its own context, workers re-read
overlapping material, the lead pays tokens to write briefs and again to read findings, and
coordination itself — planning, delegating, merging — consists of model calls. The premium implies
multi-agent only suits tasks whose value clearly covers it.
</details>

> **Q:** Why does *intrinsic* self-correction often reduce accuracy, and what still works? *(exam-relevant)*
<details><summary>Click for answer</summary>

The review pass samples from the **same model**, conditioned on substantially the same context that
produced the error — the two passes are correlated draws from one distribution, so a fact the model
wrongly believes while drafting it still believes while reviewing (**shared blind spots**; Huang et
al., 2024). What still works: surface properties the model can verify against an explicit standard —
**format, completeness against a checklist, style** — and any critique grounded in an **external
signal** (tests, retrieval, environment outcomes). Self-critique catches what the model *knows but
did not apply*, never what it *does not know*.
</details>

> **Q:** How does Self-Refine (Madaan et al., 2023) differ from Reflexion? *(not exam-relevant)*
<details><summary>Click for answer</summary>

**Self-Refine** is a single-episode loop: the same model drafts, critiques its own output, and
revises *within one task*, with no persistent memory and typically no external evaluator.
**Reflexion** operates *across episodes*: an evaluator scores whole attempts, and verbal lessons
persist in episodic memory to condition future attempts. Self-Refine fits quick quality polishing
where outputs are easy to judge; Reflexion fits repeated tasks with a reliable external success
signal.
</details>

## Part F — Stretch: two parallel workers, and the telephone game

The lecture's stretch goal (Slide 23): split the research across **two workers**, each reading
half the sources, then **merge** their findings into one report. This is *supervisor-workers* in
miniature — and it is where the **telephone game** appears. Watch what happens to the **hedges**:
some sources say things like *"a preliminary run, on a small sample, suggested ~69% (not yet
significant)"*. Every handoff is a lossy natural-language compression, and the merge step tends to
harden such hedges into flat facts — **without anyone hallucinating** (Slide 13).

In [ ]:
def worker(question, pages, worker_id):
    """A scoped sub-researcher: reads its own pages, returns compressed findings + tokens."""
    src_block = "\n".join(f"[{i}] {p['title']}: {p['body']}" for i, p in enumerate(pages, 1))
    system = ("You are a sub-researcher. Read ONLY your assigned sources and return 2-4 bullet "
              "findings, each citing a source number. Preserve any hedges (e.g. 'preliminary', "
              "'small sample') exactly — do not upgrade a hedge to a fact.")
    text, tok = chat(system, f"Question: {question}\n\nYour sources:\n{src_block}")
    return {"id": worker_id, "findings": text, "tokens": tok}


def merge(question, findings_list):
    """The lead merges two workers' findings into one report."""
    joined = "\n\n".join(f"Worker {w['id']} findings:\n{w['findings']}" for w in findings_list)
    system = ("You are the lead. Merge the workers' findings into one Markdown report with a "
              "'## Findings' and a '## Sources' section, citing source numbers.")
    text, tok = chat(system, f"Question: {question}\n\n{joined}")
    return text, tok


def two_worker_report(topic, question):
    """Supervisor-workers in miniature: split sources, run two workers, merge."""
    pages = pages_for(topic)
    left_pages  = pages[:___]      # worker 1 reads the first half of the sources
    right_pages = pages[len(pages)//2:]     # worker 2 reads the second half
    w1 = worker(question, left_pages, 1)
    w2 = worker(question, right_pages, 2)
    report, _ = merge(question, [w1, w2])
    return report, [w1, w2]


if OLLAMA_OK:
    q = TOPICS["zephyr7"]
    merged, workers = two_worker_report("zephyr7", q)
    print("=== Worker 1 raw findings ===\n", workers[0]["findings"], "\n")
    print("=== Worker 2 raw findings ===\n", workers[1]["findings"], "\n")
    print("=== Merged report ===\n", merged[:700], "...")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    left_pages  = pages[:len(pages)//2]      # worker 1 reads the first half of the sources
    right_pages = pages[len(pages)//2:]      # worker 2 reads the second half
```

Splitting the source list in half gives each worker a **disjoint slice** — the closest thing to
partitioned state (S08/S12) in this toy: two workers reading the same page and "fixing" the same
finding differently is exactly the shared-state race the lecture warns about, so we avoid overlap.

</details>

> **📝 Report task R3 — the telephone game in the merge (stretch):** Run the two-worker parallel variant in Part F. Compare the merged report against the two workers' **raw findings**: find at least one place where a hedge or nuance present in a worker's findings is **lost or hardened** in the merge (e.g. "a preliminary run on a small sample suggested" becomes "the critic lifts the score to"). Quote both versions and explain, in the lecture's terms, why every handoff is a lossy compression step.
>
> *No solution is provided — include the quoted before/after and your explanation in your lab report.*

## Part G — Tuning & exploration (no gaps — play freely)

Everything below is a playground: **no `___` gaps**. Turn the dials the lecture cares about and
watch the trade-offs move. Nothing here is graded; the point is to build intuition for *when the
second model call buys you something*.

Dials worth turning:

- **`MAX_ROUNDS`** — 0 (baseline), 1, 2, 3. Where do gains stop and cost keep climbing?
- **critic strictness** — soften/harden `CRITIC_SYSTEM`. A lenient critic passes everything (no
  value); an impossible critic loops to `unresolved` every time (pure cost).
- **source-free critic** — delete the sources from `critique` and watch the *intrinsic
  self-correction* failure mode (Huang et al., 2024): no external signal, no reliable gain.
- **worker count** — split into 3 workers instead of 2 and hunt for *more* telephone-game loss.

In [ ]:
# --- Playground: sweep MAX_ROUNDS on one topic (no gaps here) ---
def sweep_rounds(topic, question, max_list=(0, 1, 2, 3)):
    if not OLLAMA_OK:
        print("Ollama offline — start it to run the sweep."); return pd.DataFrame()
    global MAX_ROUNDS
    saved = MAX_ROUNDS
    out = []
    for m in max_list:
        MAX_ROUNDS = m
        if m == 0:
            res = run_pipeline(write_report, topic, question)
        else:
            res = run_pipeline(write_with_critic, topic, question)
        out.append({"max_rounds": m, "unsupported": res["unsupported"],
                    "tokens": res["tokens"], "latency_s": res["latency_s"]})
    MAX_ROUNDS = saved
    return pd.DataFrame(out)


# Try it (safe to re-run):
sweep_df = sweep_rounds("corvus", TOPICS["corvus"])
print(sweep_df.to_string(index=False) if not sweep_df.empty else "(offline)")

# --- Optional ipywidgets slider (graceful fallback if not installed) ---
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _show(rounds):
        if not OLLAMA_OK:
            print("Ollama offline."); return
        global MAX_ROUNDS
        MAX_ROUNDS = rounds
        res = run_pipeline(write_report if rounds == 0 else write_with_critic,
                           "halcyon", TOPICS["halcyon"])
        print(f"rounds={rounds}: unsupported={res['unsupported']:.0%}, "
              f"tokens={res['tokens']}, latency={res['latency_s']}s")

    display(widgets.interactive(_show, rounds=widgets.IntSlider(min=0, max=3, value=2)))
except Exception as exc:
    print("ipywidgets not available — use sweep_rounds(...) above instead.", f"({exc})")

> **Q:** Along what single axis does the evidence on multi-agent performance sort, and which task families sit at each end?
<details><summary>Click for answer</summary>

**Task coupling.** Loosely coupled, breadth-first tasks — literature surveys, multi-vendor
evaluations, claim-checking at scale — decompose into independent leads and benefit from parallel
contexts; this is where multi-agent **wins**. Tightly coupled tasks — canonically coding in one
shared codebase, where interfaces and invariants link every subtask — turn dependencies into lossy
handoffs with compounding errors; this is where multi-agent **loses** to a good single agent.
</details>

## Wrap-up

**Takeaways**

- The default is **one agent, one context**. Multi-agent must earn its place via **context
  separation, role specialisation, or parallelism** — and carries a token premium up to ~**15x**.
- The **minimal viable team** — a writer plus a **source-grounded critic** in a fresh context,
  with **checkable verdicts** and a **bounded** loop — captures most of the value (independent
  verification) at almost none of the cost.
- Reflection is an **amplifier of its feedback signal**: wire it to **external ground truth**
  (here, the sources) and it amplifies correction; feed it self-opinion and it amplifies blind
  spots. Self-critique catches what the model *knows but did not apply*, never what it *does not know*.
- The **telephone game** is real: every handoff is a lossy compression that can harden a hedge into
  a fact — measure the merge against the raw findings.
- You **measured** it: grounding, tokens, and latency, baseline vs. critic — numbers, not
  impressions.

**Next week — Session 13: Agent Security.** Prompt injection, the attack that works because
instructions and data share one channel, and what defence actually looks like. You will bring this
instrumented two-context agent to that session.

---

### 📝 For your lab report

Include the following (numbered as in the notebook):

- **R1** — one agent or many? Justify (or reject) a team and an agent-plus-critic for your research
  agent, naming which of the three pressures each targets and stating the burden-of-proof argument.
- **R2** — the half-page memo: *did the critic pay for itself?* Argue from your Part E table
  (grounding, tokens, latency) and give a production decision rule.
- **R3** — the telephone game in the merge: quote a hedge lost/hardened between a worker's raw
  findings and the merged report (Part F), and explain why every handoff is a lossy compression.
- **R4** — the checkable-verdict critic prompt: complete the critic's system prompt (Part C) and
  justify each choice (fresh context, external signal, strict-JSON checkable verdicts).

*(R1, R2, R3 are questions; R4 is a code gap. None has a fold-out solution in this notebook.)*